In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

In [39]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

In [3]:
df = pd.read_csv('Datasets/Student_Performance.csv')
#use below statement to convert True/False to 1/0
#df['Extracurricular Activities'] = df['Extracurricular Activities'].astype(int)
df.replace({'No': 0, 'Yes': 1}, inplace=True)

X = df.iloc[:,0:-1].values
y = df.iloc[:,-1].values

C:\Users\prajw\AppData\Local\Temp\ipykernel_20128\1009540167.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({'No': 0, 'Yes': 1}, inplace=True)


In [40]:
#normalize the data
scaler = StandardScaler()

In [41]:
X = scaler.fit_transform(X)
y = scaler.fit_transform(y)

In [42]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [43]:
regression = LinearRegression()

In [44]:
regression.fit(X_train, y_train)

LinearRegression()

In [50]:
y_pred = regression.predict(X_test)

In [8]:
print("Before normalization")
loss = ((y_pred - y_test)**2).mean()
print("Loss using OLS method is", loss)
print("Gradients are", regression.coef_, regression.intercept_)

Loss using OLS method is 4.046599085106809
Gradients are [2.85719087 1.01882142 0.63550078 0.47310588 0.18920933] -34.0698599212025


In [48]:
X_train = scaler.inverse_transform(X_train)
X_test = scaler.inverse_transform(X_test)
y_train = scaler.inverse_transform(y_train)
y_test = scaler.inverse_transform(y_test)

In [51]:
print("After data normalization")
loss = ((y_pred - y_test)**2).mean()
print("Loss using OLS method is", loss)
print("Gradients are", regression.coef_, regression.intercept_)

After data normalization
Loss using OLS method is 468.48212130322446
Gradients are [[0.38438372 0.91866935 0.01711092 0.04221533 0.02857408]] [-0.00095751]


In [9]:
#build a neural network for this 1 dependent and 5 independent variables with the use of nn.Linear

In [10]:
X = torch.tensor(X.astype(np.float32))
y = torch.tensor(y, dtype=torch.float32).reshape(-1,1)
#above reshaping of tensor y is to further match the shape with y_pred(7000,1) to esaily calculate the loss

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [22]:
class neuralNetwork(nn.Module):
    def __init__(self):
        super(neuralNetwork, self).__init__()
        #self.m = nn.Parameter(torch.zeros(5, 1))
        #self.c = nn.Parameter(torch.zeros(1,1))
        self.linear = nn.Linear(5,1)
        self.relu = nn.ReLU()

    def forward(self, x):
        #y = (x@self.m + self.c)
        y = self.linear(x)
        #y = self.relu(y)
        return y

In [33]:
regression = neuralNetwork()
epoch = 500
optim = torch.optim.SGD(regression.parameters(), lr = 0.001)

In [34]:
regression.parameters()

<generator object Module.parameters at 0x0000027D1A91B3E0>

In [35]:
for i in range(epoch):
    y_pred = regression(X_train)
    loss = ((y_pred - y_train)**2).mean()

    optim.zero_grad()
    loss.backward()
    optim.step()

    if (i%50 == 0):
        print(i, loss.item(), regression.linear.weight.grad, regression.linear.bias.grad)

0 3216.633544921875 tensor([[ -570.1404, -8032.8955,   -52.7512,  -698.6940,  -502.1069]]) tensor([-106.8082])
50 67.703857421875 tensor([[-22.3756,   0.1908,   0.8396,  13.8131,   1.1219]]) tensor([2.1490])
100 64.39866638183594 tensor([[-20.8989,   0.1224,   0.8464,  13.4700,   1.0967]]) tensor([2.1564])
150 61.438209533691406 tensor([[-19.5309,   0.0625,   0.8516,  13.1188,   1.0597]]) tensor([2.1610])
200 58.782352447509766 tensor([[-1.8263e+01,  1.1060e-02,  8.5533e-01,  1.2762e+01,  1.0135e+00]]) tensor([2.1633])
250 56.39619064331055 tensor([[-17.0870,  -0.0338,   0.8579,  12.4020,   0.9604]]) tensor([2.1635])
300 54.24930953979492 tensor([[-15.9957,  -0.0720,   0.8594,  12.0405,   0.9022]]) tensor([2.1619])
350 52.31511688232422 tensor([[-14.9823,  -0.1054,   0.8599,  11.6792,   0.8405]]) tensor([2.1587])
400 50.570281982421875 tensor([[-14.0407,  -0.1334,   0.8596,  11.3195,   0.7766]]) tensor([2.1543])
450 48.99433517456055 tensor([[-13.1653,  -0.1572,   0.8586,  10.9627,   0

In [37]:
#calculate the loss on test data
print("Before normalization")
y_pred = regression(X_test)
loss = ((y_pred - y_test)**2).mean()
print("Learning rate:", 0.0001)
print("Loss in Neural network is:",loss)
print("Gradients are:",regression.linear.weight.grad, regression.linear.bias.grad)

Before normalization
Learning rate: 0.0001
Loss in Neural network is: tensor(48.1650, grad_fn=<MeanBackward0>)
Gradients are: tensor([[-12.3667,  -0.1768,   0.8570,  10.6168,   0.6478]]) tensor([2.1423])


In [38]:
print("After data normalization")
y_pred = regression(X_test)
loss = ((y_pred - y_test)**2).mean()
print("Learning rate:", 0.001)
print("Loss in Neural network is:",loss)
print("Gradients are:",regression.linear.weight.grad, regression.linear.bias.grad)

_IncompleteInputError: incomplete input (149104261.py, line 1)